In [1]:
import os
import time
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms
from PIL import Image, ImageFile

# Allow PIL to load truncated/slightly corrupted images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# -------------------------------------------------------------------------
# 1. Configuration & Paths
# -------------------------------------------------------------------------
DATA_DIR = r"D:\Shivam Chaudhary\AI-ML Projects\Deep Learning projects\3. Document Classification with 16 Classes\Data"

BATCH_SIZE = 32
NUM_EPOCHS = 1  # Changed to 1 epoch
LEARNING_RATE = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -------------------------------------------------------------------------
# 2. Safe Image Loader (Prevents UnidentifiedImageError Crashing)
# -------------------------------------------------------------------------
def safe_pil_loader(path):
    """Safely loads images, falling back to OpenCV or a blank image if corrupted."""
    try:
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('RGB')
    except Exception:
        # Fallback 1: Try reading with OpenCV (Handles compressed TIFFs)
        try:
            img_cv = cv2.imread(path)
            if img_cv is not None:
                img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
                return Image.fromarray(img_cv)
        except Exception:
            pass
        
        # Fallback 2: Return a blank 224x224 RGB image so training doesn't crash
        print(f"Warning: Corrupted image replaced with blank canvas -> {path}")
        return Image.new('RGB', (224, 224), (0, 0, 0))

# -------------------------------------------------------------------------
# 3. Data Preprocessing & Augmentation
# -------------------------------------------------------------------------
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(degrees=5),      # Slight angle variation
        transforms.RandomHorizontalFlip(p=0.2),   # Data augmentation
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# -------------------------------------------------------------------------
# 4. Load Dataset using torchvision ImageFolder + Safe Loader
# -------------------------------------------------------------------------
print("Loading dataset...")
full_dataset = datasets.ImageFolder(root=DATA_DIR, loader=safe_pil_loader)
class_names = full_dataset.classes
print(f"Found {len(class_names)} classes: {class_names}")

# Split into 80% Training and 20% Validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_subset, val_subset = random_split(
    full_dataset, [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

# Apply respective transforms
train_subset.dataset.transform = data_transforms['train']
val_subset.dataset.transform = data_transforms['val']

# Using num_workers=0 to prevent Windows multi-processing worker issues
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# -------------------------------------------------------------------------
# 5. Define the CNN Model (ResNet-18 Transfer Learning)
# -------------------------------------------------------------------------
print("Initializing CNN Model...")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace final classification head to match target classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))

model = model.to(device)

# Loss Function & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# -------------------------------------------------------------------------
# 6. Training Loop
# -------------------------------------------------------------------------
def train_model(model, criterion, optimizer, num_epochs):
    since = time.time()

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass + optimize only in training
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

    time_elapsed = time.time() - since
    print(f"\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    return model

# -------------------------------------------------------------------------
# 7. Main Execution
# -------------------------------------------------------------------------
if __name__ == '__main__':
    trained_model = train_model(model, criterion, optimizer, num_epochs=NUM_EPOCHS)
    
    # Save trained weights
    save_path = "rvl_cdip_cnn_model.pth"
    torch.save(trained_model.state_dict(), save_path)
    print(f"Model successfully saved to {save_path}")

Using device: cpu
Loading dataset...
Found 1 classes: ['test']
Initializing CNN Model...

Epoch 1/1
------------------------------


d:\Shivam Chaudhary\Courses\Practice\cvenv\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Train Loss: 0.0000 | Acc: 1.0000
Val Loss: 0.0000 | Acc: 1.0000

Training complete in 59m 2s
Model successfully saved to rvl_cdip_cnn_model.pth
